# Balanceamento de classes

Vamos abordar classes desbalanceadas, usaremos algumas tecnicas e por ?ltimo, no SMOTE veremos a diferen?a de balancear ou n?o a valida??o.<br>
OBS: Como nosso objetivo neste tutorial ? tratar o balanceamento de dados n?o trataremos a tabela com a devida aten??o.

## Bibliotecas

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [ ]:
#!pip install imblearn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss
import seaborn as sns

## Coleta de dados
Coluna default (alvo), fraudes.

In [ ]:
df = pd.read_csv("bank-full.csv", sep=',' )
df.head(4)

## Tratamento de dados
Transformando em vari?veis dummies

In [ ]:
cols = ['education', 'default',  'housing',
       'loan', 'contact', 'month', 'poutcome', 'Target']
# Aplicando LabelEncoder para coluna facebook (0 ou 1)
for col in cols:
    encoder = LabelEncoder().fit(df[col])
    df[col] = encoder.transform(df[col])
    
# Aplicando One-hot encoder para as demais colunas (usando get_dummies)   
col_dummies = ['job', 'marital']


df = pd.get_dummies(df, columns = col_dummies)
df.head(4)   

### Treinando o modelo (sem balancear)

In [ ]:
# Verificando a discrepancia dos dados
# Notamos que est? completamente desbalanceados
df.default.value_counts()

In [ ]:
import seaborn as sns
ax = sns.countplot(x='default', data=df)

### Algoritmo classificador

In [ ]:
lr = LogisticRegression(solver='lbfgs', max_iter=100)


### Separar dados e suas classes

In [ ]:
# Separa a variavel alvo da tabela
X = df.drop('default', axis=1)
y = df.default

### Separar dados de treino e teste
    * Percentual padr?o: 70% treino 30% teste
    * random_state: Aleat?riedade
    * stratify: Propor??o iguais de classes para treino e teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1 , stratify=y)

### Treinando o algoritmo

In [ ]:
# passando pro classificador os dados de treino
lr.fit(X_train, y_train)

### Coloca as classifica??es na var y_pred

In [ ]:
y_pred = lr.predict(X_test)

### Checando a acur?cia do modelo

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f'Accuracity de {acc:,.2%}')

### Relat?rio de classifica??o
Observe o Recall

In [ ]:
print(classification_report(y_test, y_pred))

### Matriz de confus?o

In [ ]:
print(pd.crosstab(y_test, y_pred, rownames=['Real'], colnames=['Predito'], margins=True))

# Balanceamento de classes

## Aplicando NearMiss
    * M?todo de Undersampling
    * Iguala as quantidades de amostras entre as classes reduzindo a classe majorit?ria (pode haver muita perda de dados).

In [ ]:
from imblearn.under_sampling import NearMiss

### Instanciando NR

In [ ]:
# Aqui podemos passar os parametros para o nr
nr = NearMiss()

### Aplicando NR

In [ ]:
# descobre as classes majorit?rias e iguala as minoritarias
X, y = nr.fit_resample(X, y)
X.shape, y.shape

### Checando as novas amostras
    * Reduziu as classes majoritarias a n?vel de minoritaria ficando com apenas 815 amostras cada, perda extremamente considarada, mas vamos continuar...

In [ ]:
ax = sns.countplot(x=y)

### Treinando modelo (classes balanceadas)

In [ ]:
# Separando treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1 , stratify=y)
# Treinando o modelo com classificador LR
lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)
# Testando o modelo
y_pred = lr.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracity de {acc:,.2%}')

### Recall

In [ ]:
print(classification_report(y_test, y_pred))

### Matriz de confus?o

In [ ]:
print(pd.crosstab(y_test, y_pred, rownames=['Real'], colnames=['Predito'], margins=True))

NOTAS: Apesar da acuracia cair, nosso modelo conseguiu prever com melhor efici?ncia ambas as classes.

# Aplicando SMOTE
Aqui vamos criar uma fun??o que separa o validor antes de balancear as classes, assmi temos um validador fiel a realidade.


In [ ]:
from imblearn.over_sampling import SMOTE

### Verificando o desbalanceamento da coluna Target (alvo)

In [ ]:
# desbalanceamento em quantidade
df.default.value_counts()

In [ ]:
# desbalanceamento em porcentagem
100* df.default.value_counts(normalize=True)

### Separar dados e suas classes

In [ ]:
# Separa a variavel alvo da tabela
X = df.drop('default', axis=1)
y = df[['default']]

X.shape, y.shape

### Separando dados de treino e teste
- Aqui vem a sacada, separar primeiro os dados, realizar o CV, para somente depooooois realizar o balanceamento nos dados de treino.

In [ ]:
X_treino, X_test, y_treino, y_test = train_test_split(X, y, stratify=y)

X_treino.shape, X_test.shape, y_treino.shape, y_test.shape


# SMOTE  (S/ valida??o balanceada)
* Forma correta

#### Valida??o cruzada
- Vamos criar nossa valida??o cruzada na unha, pra garantir que nossa valida??o n?o esteja contaminada com dados balanceados, coisa que o cross validation n?o faz sem nossa "ajudinha"

In [ ]:
# fun??o de valida??o cruzada "artesanal"
from sklearn.model_selection import KFold

# Oversamping = False nossa valida??o n?o ser? balanceada
def validacao_cruzada(modelo, X, y, oversampling=False):
    
    kfold = KFold(n_splits=5) # separa a tabela em 5 partes
    acuracias_split = []
    
    # iterando sobres os splits
    for idx, (idx_treino, idx_validacao) in enumerate(kfold.split(X)):
        X_split_treino = X.iloc[idx_treino, :]
        y_split_treino = y.iloc[idx_treino, :]
        
        # oversampling = TRUE: balanceamento SOMENTE no split de treino
        if oversampling:
            sm = SMOTE(random_state=40)
            X_split_treino, y_split_treino, = sm.fit_resample(X_split_treino, y_split_treino)
        
        # com os dados balanceados so no treino, vamos treinar nosso modelo
        modelo.fit(X_split_treino, y_split_treino.values.flatten())
        
        X_split_validacao = X.iloc[idx_validacao, :]
        y_split_validacao = y.iloc[idx_validacao, :]
        
        # validacao SEM OVERSAMPLIG, que ? o certo!
        # amostras desbalanceadas igual mundo real
        predi??es_validacao = modelo.predict(X_split_validacao)
        
        acuracia_split = accuracy_score(y_split_validacao, predi??es_validacao)
        
        acuracias_split.append(acuracia_split)
        
        acuracia_split = acuracia_split * 100

        print(f'Acur?cia do split {idx + 1}: {acuracia_split:.2f}%')
        
    return acuracias_split
        

## Criando nosso modelo

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from statistics import mean

# Criando o modelo
modelo_hgb = HistGradientBoostingClassifier()

In [ ]:
media_acuracia_sem_smote = mean(validacao_cruzada(modelo_hgb, X_treino, y_treino, oversampling=False)) * 100

In [ ]:
media_acuracia_com_smote = mean(validacao_cruzada(modelo_hgb, X_treino, y_treino, oversampling=True)) * 100

In [ ]:
f'Sem smote: {media_acuracia_sem_smote:.2f}%, com_smote: {media_acuracia_com_smote:.2f}%'

# SMOTE  (C/ valida??o balanceada)
* Forma errada

In [ ]:
from sklearn.model_selection import cross_val_score

# ERRADO! N?o se deve balancear as classes antes do split
sm = SMOTE(random_state=42)
X_treino, y_treino = sm.fit_resample(X_treino, y_treino)

# Os splits criados no cross_val_score est?o com a valida??o BALANCEADA
# E isso ? distorcer a realidade!
media = mean(cross_val_score(modelo_hgb, X_treino, y_treino.values.flatten())) * 100
print(f'M?dia com o balanceamento da valida??o: {media:.2f}%')

# Conclus?o
- Com a valida??o balanceada tivemos uma acur?cia levemente melhor, por?m o modelo n?o esta preparado para omundo real, sem balanceamento na valida??o a acur?cia pode diminuir, por?m na vida real o modelo ser? mais acertivo.
- Ent?o concluimos que nem sempre uma acur?cia mais alta ? garantia de acertividade com dados externos.